# 03 - Gemini com Python

**Gemini** é a família de modelos multimodais do Google DeepMind.
Suporta texto, imagens, áudio e vídeo nativamente, com janelas de contexto muito grandes.

### Principais modelos

| Modelo | Contexto | Uso recomendado |
|--------|----------|-----------------|
| `gemini-2.0-flash` | 1M tokens | Velocidade, uso geral — **recomendado** |
| `gemini-1.5-pro` | 2M tokens | Tarefas complexas, documentos longos |
| `gemini-1.5-flash` | 1M tokens | Leve e rápido |

> **`gemini-1.5-flash`** usado no original ainda funciona, mas `gemini-2.0-flash` é a versão atual recomendada.

### Pacotes necessários

```bash
pip install google-generativeai python-dotenv httpx
```

> Crie sua API key em [aistudio.google.com](https://aistudio.google.com) e adicione `GEMINI_API_KEY=sua_chave` no `.env`.

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
import google.generativeai as genai

load_dotenv(find_dotenv())

api_key = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)

In [9]:
import google.generativeai as genai

for model in genai.list_models():
    # filtrando só os que suportam geração de conteúdo
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

## 1. Chat básico

A API do Gemini usa `GenerativeModel` + `generate_content()`.
É diferente do padrão OpenAI — não há `client.chat.completions.create()`.

In [10]:
import textwrap

def formatar_texto(texto: str, largura: int = 100) -> None:
    print(textwrap.fill(texto, width=largura))

model = genai.GenerativeModel("models/gemini-2.5-flash")

response = model.generate_content(
    "Qual dica você recomenda para ter uma vida boa?"
)

formatar_texto(response.text)

Que ótima pergunta! Se eu tivesse que escolher apenas uma dica fundamental, seria esta:  **A dica
principal: Conheça a si mesmo e viva em alinhamento com seus valores.**  **Por que esta dica é tão
poderosa?**  1.  **Autenticidade:** Quando você se conhece de verdade – seus pontos fortes, suas
fraquezas, suas paixões, o que te motiva e o que te drena – você pode tomar decisões que são
genuinamente suas. 2.  **Propósito:** Seus valores são a sua bússola moral. Saber o que é importante
para você (seja família, liberdade, criatividade, serviço, segurança, aventura) permite que você
direcione sua energia para o que realmente importa, criando uma vida com mais sentido. 3.
**Contentamento Duradouro:** Buscar a felicidade em coisas externas ou em aprovação alheia é uma
corrida sem fim. Viver de acordo com seus próprios valores, mesmo que seja desafiador, traz uma
sensação de paz e satisfação muito mais profunda e duradoura. Você se sente no controle da sua
própria narrativa. 4.  **Resiliência:

## 2. Streaming

Com `stream=True`, os chunks chegam progressivamente.
Acumulamos o texto completo para uso posterior.

In [11]:
stream = model.generate_content(
    "Qual o sentido da felicidade?",
    stream=True
)

resposta_completa = ""
for chunk in stream:
    print(chunk.text, end="", flush=True)
    resposta_completa += chunk.text

O sentido da felicidade é um dos questionamentos mais antigos e profundos da humanidade, e não há uma resposta única e universal para ele. A felicidade é algo profundamente pessoal, subjetivo e multifacetado, que pode ser interpretado de diversas maneiras:

1.  **Um Estado de Bem-Estar e Contentamento:** Para muitos, a felicidade é um estado emocional duradouro de bem-estar, satisfação e contentamento com a vida. Não é a ausência de problemas, mas a capacidade de lidar com eles e ainda assim sentir alegria, paz e propósito.

2.  **A Busca por Propósito e Significado:** A felicidade muitas vezes não é apenas um sentimento, mas o resultado de viver uma vida com propósito. Ter metas que valem a pena, fazer a diferença, contribuir para algo maior do que si mesmo pode trazer uma profunda sensação de realização e, consequentemente, felicidade. (Essa ideia se alinha com a **Eudaimonia** de Aristóteles, que via a felicidade não como mero prazer, mas como uma vida vivida de forma virtuosa e com

## 3. Análise de Imagens

O Gemini é nativo multimodal — imagem e texto são enviados juntos na mesma chamada.
A imagem pode ser uma URL (via `httpx`) ou um arquivo local.

> **Formato esperado**: os dados da imagem devem ser enviados como base64 com o `mime_type` correspondente.

In [14]:
import httpx
import base64

# Usando gemini-1.5-pro para análise de imagem (melhor capacidade visual)
model_vision = genai.GenerativeModel(model_name="models/gemini-2.5-pro")

image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg/2560px-Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg"

# Baixando a imagem e convertendo para base64
image_data = httpx.get(image_url)

response = model_vision.generate_content([
    {
        "mime_type": "image/jpeg",
        "data": base64.b64encode(image_data.content).decode("utf-8")
    },
    "Descreva a imagem para mim"
])

formatar_texto(response.text)

InvalidArgument: 400 Unable to process input image. Please retry or report in https://developers.generativeai.google/guide/troubleshooting

## 4. Histórico de Conversa

O Gemini tem suporte nativo a histórico com `start_chat(history=[])`.
Cada chamada a `send_message()` adiciona automaticamente ao histórico.

É diferente da OpenAI, onde você gerencia o histórico manualmente passando todas as mensagens a cada chamada.

| Abordagem | Gemini nativo | OpenAI / LangChain |
|-----------|--------------|--------------------|
| Histórico | automático via `chat.history` | manual — você mantém a lista |
| Envio | `chat.send_message(texto)` | `client.chat.completions.create(messages=[...])` |

In [16]:
model_chat = genai.GenerativeModel("gemini-2.5-flash")

# history=[] inicia uma conversa sem contexto anterior
chat_session = model_chat.start_chat(history=[])

# Primeira mensagem
response1 = chat_session.send_message("Explique o que é LangChain para uma criança")
formatar_texto(response1.text)
print()

Imagina que você tem um amigo **super, super inteligente** que sabe MUITO sobre palavras. Ele
consegue te contar histórias incríveis, responder a quase todas as suas perguntas e até escrever
poemas bonitos. Esse amigo é como um "Cérebro Mágico que Fala" (na verdade, a gente chama de Modelo
de Linguagem Grande ou LLM).  Mas, às vezes, mesmo o amigo mais inteligente precisa de ajuda para
fazer algumas coisas. Por exemplo:  1.  Se você perguntar: "Qual é a temperatura em Paris agora?",
seu amigo sabe *muito* sobre palavras, mas ele não consegue *olhar na internet* para ver o tempo em
Paris. 2.  Se você pedir: "Me ajude a planejar uma festa de aniversário", ele pode te dar ideias,
mas talvez não consiga *organizar tudo em passos* ou *mandar convites*. 3.  Se você conversar muito
com ele, pode ser que ele *esqueça* o que vocês falaram no começo.  **LangChain é como um "kit de
ferramentas mágico" para o seu amigo super inteligente!**  Com o LangChain, seu Cérebro Mágico que
Fala consegue:  *

In [17]:
# Segunda mensagem — o modelo lembra da conversa anterior
response2 = chat_session.send_message("Agora explique para um adulto")
formatar_texto(response2.text)

Para um adulto, podemos descrever LangChain de uma forma mais técnica e focada na sua utilidade no
desenvolvimento de aplicações.  ---  **LangChain é um framework de código aberto que simplifica o
desenvolvimento de aplicações complexas e data-driven baseadas em Large Language Models (LLMs).**
Em essência, os LLMs (como GPT-4, Claude, Llama) são modelos incrivelmente poderosos na geração e
compreensão de texto. No entanto, eles possuem algumas limitações intrínsecas ao serem usados em
aplicações práticas:  1.  **Statelessness (Falta de Memória Persistente):** Um LLM, por si só, não
"lembra" de conversas passadas ou de informações que lhe foram dadas em interações anteriores dentro
de uma mesma sessão. Cada chamada é uma interação nova. 2.  **Falta de Conhecimento Externo e Tempo
Real:** O conhecimento de um LLM é "congelado" na data de seu último treinamento. Ele não pode
acessar a internet em tempo real, bases de dados corporativas, APIs externas ou realizar cálculos
complexos. 3.  **

In [18]:
# Inspecionando o histórico completo
print(f"Total de turnos no histórico: {len(chat_session.history)}\n")
for msg in chat_session.history:
    role = msg.role.upper()
    texto = msg.parts[0].text[:100].replace("\n", " ")
    print(f"[{role}] {texto}...")

Total de turnos no histórico: 4

[USER] Explique o que é LangChain para uma criança...
[MODEL] Imagina que você tem um amigo **super, super inteligente** que sabe MUITO sobre palavras. Ele conseg...
[USER] Agora explique para um adulto...
[MODEL] Para um adulto, podemos descrever LangChain de uma forma mais técnica e focada na sua utilidade no d...


## Resumo

| Funcionalidade | Método |
|---------------|--------|
| Chat simples | `model.generate_content(texto)` |
| Streaming | `model.generate_content(texto, stream=True)` |
| Análise de imagem | `generate_content([{mime_type, data}, prompt])` |
| Chat com histórico | `model.start_chat()` + `chat.send_message()` |
| Ver histórico | `chat.history` |

> **Próximo passo**: usar o Gemini com LangChain via `ChatGoogleGenerativeAI`
> para integrar com chains, agentes e RAG.